# Wildfire Comprehensive Notebook Flow

This notebook consolidates the existing wildfire notebooks into one flow with:
1) aggregated imports, 2) data loading, and 3) combined EDA + processing.

In [ ]:
# Aggregated imports from existing notebooks
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    silhouette_score,
)
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

# Optional dependencies used in other notebooks
try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    SMOTE = None

try:
    from autogluon.tabular import TabularDataset, TabularPredictor
except ImportError:
    TabularDataset = None
    TabularPredictor = None

try:
    from lightgbm import LGBMRegressor
except ImportError:
    LGBMRegressor = None

try:
    import joblib
except ImportError:
    joblib = None

try:
    import kagglehub
except ImportError:
    kagglehub = None

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 100)
print('Imports loaded (optional libraries may be unavailable).')


In [ ]:
# Data loading (supports both file names used in existing notebooks)
repo_root = Path.cwd()
candidates = [repo_root / 'wildfire.csv', repo_root / 'final_dataset.csv']

data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not find wildfire.csv or final_dataset.csv in the working directory.')

df = pd.read_csv(data_path)
print(f'Loaded: {data_path.name}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()


In [ ]:
# Combined EDA + processing cell
print('--- EDA: Dataset Health ---')
print(df.dtypes)
print('\nMissing values by column:')
print(df.isna().sum().sort_values(ascending=False).head(15))
print(f"\nDuplicate rows: {df.duplicated().sum()}")

DEFAULT_N_CLUSTERS = 4
MIN_N_CLUSTERS = 2

def clamp_cluster_count(unique_labels):
    if unique_labels <= 1:
        return DEFAULT_N_CLUSTERS
    return min(DEFAULT_N_CLUSTERS, max(MIN_N_CLUSTERS, unique_labels))

def find_occurrence_target_column(columns):
    return next((c for c in columns if c.lower().startswith('occur')), None)

target_col = find_occurrence_target_column(df.columns)
if target_col is not None:
    print(f'\nTarget distribution ({target_col}):')
    print(df[target_col].value_counts(dropna=False))

# Basic visual EDA for numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    corr = df[numeric_cols].corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, cmap='coolwarm', center=0)
    plt.title('Numeric Feature Correlation Heatmap')
    plt.tight_layout()
    plt.show()

print('\n--- Processing ---')
work_df = df.copy()
work_df = work_df.drop_duplicates()

# Feature engineering used in existing workflows
if 'frp' in work_df.columns and 'frp_log' not in work_df.columns:
    work_df['frp_log'] = np.log1p(work_df['frp'].clip(lower=0))

# Common target used in classification notebooks
if target_col is not None:
    model_df = work_df.dropna(subset=[target_col]).copy()
    X = model_df.drop(columns=[target_col])
    y = model_df[target_col]

    # Use numeric-only for a portable baseline preprocessing flow
    X = X.select_dtypes(include=[np.number]).copy()

    has_multiple_classes = y.nunique() > 1
    stratify_by = y if has_multiple_classes else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=stratify_by
    )

    if SMOTE is not None and has_multiple_classes:
        sm = SMOTE(random_state=42)
        X_train, y_train = sm.fit_resample(X_train, y_train)
        print('Applied SMOTE to training split.')
    else:
        print('SMOTE unavailable or not applicable; skipping oversampling.')

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f'Processed train shape: {X_train.shape}, test shape: {X_test.shape}')

    # Optional unsupervised processing inspired by clustering notebook
    if X.shape[0] >= 10 and X.shape[1] >= 2:
        unique_labels = y.nunique()
        # Clamp cluster count to a stable, interpretable range.
        n_clusters = clamp_cluster_count(unique_labels)
        km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        km_labels = km.fit_predict(X)
        if len(np.unique(km_labels)) > 1:
            sil = silhouette_score(X, km_labels)
            print(f'KMeans silhouette score (k={n_clusters}): {sil:.4f}')

        gmm = GaussianMixture(n_components=n_clusters, covariance_type='full', random_state=42)
        gmm_labels = gmm.fit_predict(X)
        print(f'GMM clusters created: {len(np.unique(gmm_labels))}')

else:
    print("Target column not found (expected a column starting with 'occur'). EDA completed; supervised processing skipped.")

print('\nFlow complete.')
